# RDKit SDF Deduplication

This notebook demonstrates the SDF/MOL path in `deduplicate_rdkit.py`. Use this path when inputs already contain explicit molecular connectivity.

In [ ]:
from pathlib import Path
import subprocess
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "deduplicate_rdkit.py").is_file():
            return candidate
    raise RuntimeError("Could not find repository root")

repo = find_repo_root(Path.cwd().resolve())
src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from rdkit.Chem import rdMolAlign
from deduplicate_rdkit import read_sdf, renumber_to_template, mol_name

data = src / "testdata"
sdf_path = data / "ethanol_reordered.sdf"
sdf_path

## Read SDF records

`read_sdf` returns `(path, molecule_index, mol)` tuples and preserves explicit hydrogens.

In [ ]:
records = read_sdf([sdf_path])

print("records:", len(records))
for path, mol_index, mol in records:
    print(mol_index, mol_name(mol, f"mol_{mol_index}"), "atoms=", mol.GetNumAtoms())

## Renumber to the template graph

The helper maps every molecule onto the atom order of the first molecule by full graph match. After renumbering, symmetry-aware RDKit RMSD can compare conformers robustly.

In [ ]:
template = records[0][2]
aligned = []
for path, mol_index, mol in records:
    aligned_mol = renumber_to_template(template, mol)
    aligned.append(aligned_mol)
    print(mol_index, mol_name(aligned_mol, ""), [atom.GetSymbol() for atom in aligned_mol.GetAtoms()])

rmsd = rdMolAlign.GetBestRMS(aligned[0], aligned[1])
print("RMSD between first two SDF records:", f"{rmsd:.6f}")

## Run the command-line script

The script reports counts and can write unique representatives to a new SDF.

In [ ]:
outdir = repo / "tutorials" / "output"
outdir.mkdir(parents=True, exist_ok=True)
unique_sdf = outdir / "ethanol_unique.sdf"

completed = subprocess.run(
    [
        sys.executable,
        str(src / "deduplicate_rdkit.py"),
        str(sdf_path),
        "--tolerance",
        "0.001",
        "--write-unique",
        str(unique_sdf),
    ],
    text=True,
    capture_output=True,
    check=True,
)

print(completed.stdout)
print("wrote:", unique_sdf)

## Read the unique output

The output SDF stores source metadata on each unique representative.

In [ ]:
unique_records = read_sdf([unique_sdf])
for path, mol_index, mol in unique_records:
    print(
        mol_index,
        mol_name(mol, ""),
        mol.GetProp("source_path"),
        mol.GetIntProp("source_molecule_index"),
    )